# Filing chunk and retrieval check

This notebook checks three practical things: whether the filing chunks fit the embedding model, whether their source information is present, and whether search returns the expected evidence.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

In [2]:
from chunking import build_filing_chunks, load_filing_sources
from config import QDRANT_DENSE_MAX_TOKENS, QDRANT_DENSE_MODEL
from embeddings import build_chunk_token_report
from retrieval import search_filing

/home/atanas-vitanov/Documents/final_project_financial_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Model used for search

In [3]:
{"model": QDRANT_DENSE_MODEL, "max_tokens": QDRANT_DENSE_MAX_TOKENS}

{'model': 'snowflake/snowflake-arctic-embed-m-long', 'max_tokens': 2048}

## Build the chunks and count their tokens

In [4]:
chunk_records, summaries = build_chunk_token_report()

## Chunk size overview

In [5]:
summaries

[{'ticker': 'AMZN',
  'chunk_count': 142,
  'average_tokens': 767.4,
  'max_tokens': 1494,
  'largest_chunk': 'filing-amzn-2025-page-0023',
  'model_limit': 2048,
  'within_model_limit': True},
 {'ticker': 'GOOGL',
  'chunk_count': 179,
  'average_tokens': 745.6,
  'max_tokens': 1541,
  'largest_chunk': 'filing-googl-2025-page-0053-table-02',
  'model_limit': 2048,
  'within_model_limit': True},
 {'ticker': 'KO',
  'chunk_count': 248,
  'average_tokens': 862.7,
  'max_tokens': 1639,
  'largest_chunk': 'filing-ko-2025-page-0135',
  'model_limit': 2048,
  'within_model_limit': True},
 {'ticker': 'ALL',
  'chunk_count': 569,
  'average_tokens': 802.1,
  'max_tokens': 1639,
  'largest_chunk': 'filing-ko-2025-page-0135',
  'model_limit': 2048,
  'within_model_limit': True}]

## Largest chunks to inspect

In [6]:
largest_chunks = sorted(
    chunk_records, key=lambda row: row["token_count"], reverse=True
)[:10]
largest_chunks

[{'ticker': 'KO',
  'evidence_id': 'filing-ko-2025-page-0135',
  'evidence_type': 'filing_page',
  'supporting_page_sequences': [135],
  'word_count': 1083,
  'token_count': 1639},
 {'ticker': 'KO',
  'evidence_id': 'filing-ko-2025-page-0093',
  'evidence_type': 'filing_page',
  'supporting_page_sequences': [93],
  'word_count': 1275,
  'token_count': 1580},
 {'ticker': 'KO',
  'evidence_id': 'filing-ko-2025-page-0033',
  'evidence_type': 'filing_page',
  'supporting_page_sequences': [33],
  'word_count': 1233,
  'token_count': 1573},
 {'ticker': 'KO',
  'evidence_id': 'filing-ko-2025-page-0128',
  'evidence_type': 'filing_page',
  'supporting_page_sequences': [128],
  'word_count': 1058,
  'token_count': 1554},
 {'ticker': 'KO',
  'evidence_id': 'filing-ko-2025-page-0135-table-01',
  'evidence_type': 'filing_table',
  'supporting_page_sequences': [135],
  'word_count': 994,
  'token_count': 1549},
 {'ticker': 'KO',
  'evidence_id': 'filing-ko-2025-page-0129',
  'evidence_type': 'filin

## Can every chunk be embedded safely?

In [7]:
all(row["token_count"] <= QDRANT_DENSE_MAX_TOKENS for row in chunk_records)

True

**What this means:** If this cell is `True`, every chunk fits in the embedding model. No filing text is cut off before it is indexed.

## Do chunks keep the information we need?

In [8]:
chunks = [
    chunk
    for source in load_filing_sources()
    for chunk in build_filing_chunks(source)
]
payloads = [chunk.payload() for chunk in chunks]
empty_metadata = [
    (payload["evidence_id"], key)
    for payload in payloads
    for key, value in payload.items()
    if value is None or value == "" or value == [] or value == {}
]
chunk_metrics = {
    "chunk_count": len(chunks),
    "payloads_with_empty_metadata": len(
        {evidence_id for evidence_id, _ in empty_metadata}
    ),
    "empty_metadata_value_count": len(empty_metadata),
}
chunk_metrics

{'chunk_count': 569,
 'payloads_with_empty_metadata': 0,
 'empty_metadata_value_count': 0}

**What this means:** Every chunk has the metadata fields this project expects. The evidence ID and source details let us trace a search result back to the frozen filing.

## Does search find the evidence we expect?

We test only questions that use `search_filing`. Amazon and Alphabet are development and validation data. Coca-Cola is held-out test data, so its result is shown separately and is not used to change the retrieval strategy. Ragas compares evidence IDs; no model judges the answers.

In [9]:
import json
from ragas import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall

questions = [
    json.loads(line)
    for ticker in ("AMZN", "GOOGL", "KO")
    for line in (project_root / "data" / "evaluation" / ticker / "questions.jsonl")
    .read_text()
    .splitlines()
]
questions = [
    question for question in questions if "search_filing" in question["required_tools"]
]

In [10]:
recall_metric = IDBasedContextRecall()
precision_metric = IDBasedContextPrecision()
retrieval_rows = []
for question in questions:
    results = search_filing(question["question"], question["ticker"], 5)
    retrieved_ids = [result.evidence_id for result in results]
    recall_sample = SingleTurnSample(
        retrieved_context_ids=retrieved_ids,
        reference_context_ids=question["gold_retrieval_evidence_ids"],
    )
    precision_sample = SingleTurnSample(
        retrieved_context_ids=retrieved_ids,
        reference_context_ids=question["relevant_retrieval_evidence_ids"],
    )
    retrieval_rows.append(
        {
            "question_id": question["question_id"],
            "split": question["split"],
            "retrieved_evidence_ids": retrieved_ids,
            "evidence_recall_at_5": await recall_metric.single_turn_ascore(recall_sample),
            "irrelevant_retrieval_rate_at_5": 1
            - await precision_metric.single_turn_ascore(precision_sample),
        }
    )

In [11]:
retrieval_rows

[{'question_id': 'amzn-03',
  'split': 'development',
  'retrieved_evidence_ids': ['filing-amzn-2025-page-0027-table-01',
   'filing-amzn-2025-page-0068-table-01',
   'filing-amzn-2025-page-0027',
   'filing-amzn-2025-page-0018',
   'filing-amzn-2025-page-0001-table-03'],
  'evidence_recall_at_5': 1.0,
  'irrelevant_retrieval_rate_at_5': 0.6},
 {'question_id': 'amzn-04',
  'split': 'development',
  'retrieved_evidence_ids': ['filing-amzn-2025-page-0027-table-01',
   'filing-amzn-2025-page-0027',
   'filing-amzn-2025-page-0025-table-01',
   'filing-amzn-2025-page-0067',
   'filing-amzn-2025-page-0001-table-03'],
  'evidence_recall_at_5': 1.0,
  'irrelevant_retrieval_rate_at_5': 0.6},
 {'question_id': 'amzn-12',
  'split': 'development',
  'retrieved_evidence_ids': ['filing-amzn-2025-page-0025-table-01',
   'filing-amzn-2025-page-0025',
   'filing-amzn-2025-page-0024-table-01',
   'filing-amzn-2025-page-0027-table-01',
   'filing-amzn-2025-page-0024'],
  'evidence_recall_at_5': 1.0,
  'i

In [12]:
metric_groups = {
    "development_and_validation": [
        row for row in retrieval_rows if row["split"] != "held_out"
    ],
    "held_out_test": [
        row for row in retrieval_rows if row["split"] == "held_out"
    ],
}
retrieval_metrics = {
    name: {
        "evidence_recall_at_5": sum(
            row["evidence_recall_at_5"] for row in rows
        )
        / len(rows),
        "irrelevant_retrieval_rate_at_5": sum(
            row["irrelevant_retrieval_rate_at_5"] for row in rows
        )
        / len(rows),
    }
    for name, rows in metric_groups.items()
}
retrieval_metrics

{'development_and_validation': {'evidence_recall_at_5': 1.0,
  'irrelevant_retrieval_rate_at_5': 0.6499999999999999},
 'held_out_test': {'evidence_recall_at_5': 0.75,
  'irrelevant_retrieval_rate_at_5': 0.8500000000000001}}

**How to read the result:** Evidence Recall@5 asks, “did the expected page or table appear in the first five results?” Higher is better. Irrelevant Retrieval Rate@5 asks, “how many of the five results are outside the short approved list?” Lower is better. Most development and validation questions approve one page and one table, so their five-result search cannot go below `0.60` without duplicates. Some held-out questions approve only one chunk, making their lowest possible rate `0.80`. Keep the held-out result separate: it tells us how the already chosen strategy performs on Coca-Cola.

## Look at two real search results

The first question is broad. The second uses table labels and dates, so it is more likely to return a table or a table that continues onto another page. Open the payloads below to see what search actually returned.

In [13]:
inspection_queries = {
    "narrative": ("AMZN", "What drove Amazon operating income in 2025?"),
    "table": (
        "AMZN",
        "cash equivalents restricted cash and marketable securities fair value hierarchy by major investment type December 31 2024 2025 amortized cost unrealized gains losses",
    ),
}

In [14]:
inspection_results = {}
for label, (ticker, query) in inspection_queries.items():
    results = search_filing(query, ticker, 5)
    inspection_results[label] = [
        {"rank": rank, "score": result.score, "payload": result.metadata}
        for rank, result in enumerate(results, start=1)
    ]
inspection_results

{'narrative': [{'rank': 1,
   'score': 0.79545456,
   'payload': {'evidence_id': 'filing-amzn-2025-page-0027-table-01',
    'evidence_type': 'filing_table',
    'chunk_schema_version': '3',
    'document_type': '10-K',
    'company': 'AMAZON COM INC',
    'ticker': 'AMZN',
    'cik': '0001018724',
    'accession_number': '0001018724-26-000004',
    'report_year': 2025,
    'report_date': '2025-12-31',
    'filing_date': '2026-02-06',
    'source_url': 'https://www.sec.gov/Archives/edgar/data/1018724/000101872426000004/amzn-20251231.htm',
    'source_path': 'data/raw/AMZN/10-K-2025.html',
    'primary_document': 'amzn-20251231.htm',
    'source_sha256': 'beb1bfc28558135e76882db0cf36ba72a1ab44e2144fe37534f4da1e120ddb57',
    'parser_version': '3',
    'printed_page': 27,
    'supporting_page_ids': ['filing-amzn-2025-page-0027'],
    'supporting_page_sequences': [27],
    'section_title': 'Operating Income',
    'table_fragment_ids': ['filing-amzn-2025-page-0027-table-01'],
    'core_text

**What to look for:** `evidence_type` tells you whether the result is a page, a table, or a cross-page table. `supporting_page_sequences` tells you where it came from. `fusion_score` is the original search score; `linked_page_score` shows any extra help a table received from its source page.